# E1.2 · Building the AI and agent inventory

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.1 · Why point-in-time control testing fails for AI](https://spbreed.github.io/cyber-commons/lessons/E1.1.html)**.

| | |
|---|---|
| Tools used | agentgateway, SPIRE |

## What this lesson is

**What it covers.** Discover agents from gateway and identity telemetry; build the register.

**Why a security engineer needs it.** Shadow AI and shadow agents — the inventory is the control most orgs still lack. The control it builds is: discovery, registration, ownership, risk tiering.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Nobody can govern what nobody has listed. The inventory is the least interesting artefact in this function and the one everything else depends on — and the hard part is not building it, it is keeping it true next quarter.

> **At CyberTravels.** Nobody at CyberTravels can currently list every agent, MCP server and vector index in the estate. Everything else in this chapter depends on that list being true next quarter.

## 2 · The framework

```
   what has to be in the register

   model . agent . integration . MCP server . eval corpus
        |
   +----v-------------------------------------------+
   | owner . purpose . data classes . autonomy level|
   | tools it holds . environment . last verified   |
   +------------------------------------------------+

   building it is a project. keeping it true is the control.
```

You cannot govern, tier, test or revoke what you cannot list. The AI inventory is
therefore the first control, not a documentation exercise.

The honest finding of every first inventory is the same: **most of it was already
in production.** Not because anyone was reckless, but because AI features arrive
inside products you already bought, and agents get created programmatically by
other agents.

Three sources, and the third finds what the first two miss:

1. the **model registry** — what your ML team registered,
2. **procurement and expense** — what someone bought,
3. **egress logs to model-provider domains** — what is actually being used.

Source 3 is the one that discovers the department using a frontier API on a
personal card, and the SaaS product that quietly added an AI feature.

## 3 · Resolving one row into a deployment, as a skill

Discovery finds that a thing exists. Governing it needs the row resolved into artefacts: which repository at which commit, which image **digest** rather than which tag, which IAM role and SPIFFE ID, which gateway and guardrail, and every downstream its tools call. Every other attestation skill consumes this graph, which is why it runs first. This is the file in this repository:

### The skill — [`skills/attestation/deployment-inventory-resolver/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/deployment-inventory-resolver/SKILL.md)

```yaml
name: deployment-inventory-resolver
description: >-
  Resolve a deployment_id into the artifacts that make up one agentic
  deployment and build the evidence graph every other attestation skill
  consumes. Use first, before any control is evaluated, or when asked which
  repo, image, role, workload identity, gateway or guardrail a deployment
  actually consists of.
allowed-tools: Bash, Read, Grep, Glob
```

# Deployment Inventory Resolver

**Controls:** All — this skill produces the join key

## Why this runs first

`deployment_id` is the primary key for every other skill in this set. Without
it, an IAM finding, a SPIFFE entry and a gateway policy are three unrelated
facts about three things that may or may not be the same system.

This skill turns that ID into a manifest of content-addressed artifacts, so
every later finding resolves to one evaluable unit and cannot be silently
confused with a neighbouring deployment.

## Procedure

1. **Resolve the code.** Repository URL plus the exact commit SHA that was
   built. Not a branch — a branch moves.
2. **Resolve the image.** Registry digest (`sha256:…`), not a tag. Confirm the
   digest matches the revision actually deployed, not the newest build.
3. **Resolve the runtime identity.** The IAM role ARN, the SPIFFE ID, and —
   on platforms that auto-create one — the workload identity ARN exposed by the
   runtime/gateway description API.
4. **Resolve the traffic path.** Gateway or route ARN, and the guardrail ID
   attached to it.
5. **Resolve the downstreams.** Every service the deployment's tools call.
   These become the input to the risk-registry skill.
6. **Cross-link and check completeness.** Every artifact must reference the
   others. Report orphans rather than omitting them.

## Output contract

```json
{
  "deployment_id": "str",
  "resolved_at": "str",
  "artifacts": {
    "repo": {"url": "str", "commit": "str"},
    "image": {"registry": "str", "digest": "str", "matches_deployed": true},
    "identity": {"role_arn": "str", "spiffe_id": "str", "workload_identity_arn": "str"},
    "traffic": {"gateway_arn": "str", "guardrail_id": "str"},
    "downstreams": ["str"]
  },
  "missing": ["str"],
  "orphans": ["str"],
  "verdict": "PASS|PARTIAL|FAIL"
}
```

A manifest with entries in `missing` is `PARTIAL` at best. Every downstream
skill inherits that ceiling — you cannot attest a control on an artifact you
could not resolve.

## Failure modes

- **Resolving a tag instead of a digest.** Tags are mutable; the thing you
  attested is not necessarily the thing running.
- **Treating an unresolvable artifact as absent.** "No gateway configured" and
  "I could not read the gateway API" are different findings.
- **Reusing a manifest across runs.** Re-resolve. Drift is the point.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/deployment-inventory-resolver/scripts/deployment_inventory_resolver.py
SCRIPT = "skills/attestation/deployment-inventory-resolver/scripts/deployment_inventory_resolver.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The skill loads and reports its shape. Two of its rules are what make an inventory hold: resolve digests rather than tags, because a tag is mutable and the thing you attested is not the thing running; and never record an unresolvable artefact as absent — "no gateway configured" and "could not read the gateway" are different findings with different owners.

## Your turn

Run the egress query for real: one week of traffic to model-provider domains, joined against your inventory. It takes an hour and it always finds something.

---

**Next → [E1.3 · Risk tiering agentic use cases](https://spbreed.github.io/cyber-commons/lessons/E1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*